# Preview: D4 augmentation + a hidden FC layer

Branch `experiment/d4-fc-head`. **Nothing here touches the reported 56 runs**, and nothing is
written inside the git tree - every output goes to Drive and Ido moves what he wants into git.

Two changes, run together on purpose:

- **D4 augmentation** - uniform k x 90 deg rotation on top of the existing horizontal flip,
  i.e. all eight elements of the dihedral group. Exact: `rot90` is a transpose plus a
  reversal, so it permutes the pixel grid without interpolating. That is the only reason it
  is admissible in a study about resampling.
- **A hidden layer in the classifier head** - `Linear(256->256) -> ReLU -> Dropout` before the
  output layer, +65,792 parameters.

They belong together. The model is **variance-limited** (train 0.967-0.982 against val
0.905-0.910), so extra capacity on its own would normally make overfitting worse. The
augmentation is what buys the room for it.

## The control is not optional

The reported runs are **40 epochs**. D4 needs **80** - under uniform D4 only one presentation
in eight is upright, and `best_epoch` already averaged 33 of 40 without it. But the schedule is
`CosineAnnealingLR` spanning exactly `epochs`, so 80 epochs is a *different learning-rate
schedule*, not simply more of the same.

So condition A below is flip-only at 80 epochs with the original head. Without it, any gain
could just be the longer schedule, and the run would prove nothing.

| | epochs | augmentation | head |
|---|---|---|---|
| **A - control** | 80 | flip only | single linear |
| **B - experiment** | 80 | D4 | + 256 hidden |

3 sources x 2 conditions x 1 seed = **6 runs, ~25 minutes**. If B does not beat A here, do not
spend the full 28.

## Setup

Set `OWNER`. Everything else is derived, and every write lands under `EXP` in Drive.

In [ ]:
# Re-run this after any runtime restart.
import os, sys, json, glob, time
from pathlib import Path

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU on this runtime -> Runtime > Change runtime type > T4 GPU > Save, then re-run this cell (the restart unmounts Drive)"

from google.colab import drive
drive.mount('/content/drive')

OWNER = "ido"

# Read root, shared between both accounts.
DRIVE = "/content/drive/MyDrive/university/deep_learning"
CACHE = f"{DRIVE}/cache"

# EVERY write goes here. Deliberately NOT inside the cloned repo: these are experimental
# runs, they are not part of the reported results, and Ido decides by hand what enters git.
# `src.train` also refuses outright to write experimental runs into `results/runs`.
EXP = f"/content/drive/MyDrive/deep_learning_results/{OWNER}/experiments"
CTRL_DIR = f"{EXP}/ctrl80"     # condition A
D4_DIR = f"{EXP}/d4_fc256"     # condition B

os.makedirs(CTRL_DIR, exist_ok=True)
os.makedirs(D4_DIR, exist_ok=True)

assert os.path.isdir(CACHE), f"cache not found: {CACHE}"
print("cache <-", CACHE)
print("out   ->", EXP)

## Get the code — the experiment branch, not main

In [ ]:
# Idempotent. Note the branch: main does not have --d4 or --fc-hidden.
![ -d /content/deep-learning ] || git clone -q https://github.com/noa-keter/deep-learning.git /content/deep-learning
%cd /content/deep-learning
!git fetch -q origin
!git checkout experiment/d4-fc-head
!git pull --ff-only
!git log --oneline -1

# The architecture self-check: shapes, both parameter counts, and an overfit-ten-examples
# test that would fail on a broken gradient path. No data needed, runs in seconds.
!python src/model.py

## The preview — 6 runs

Three sources chosen to span the difficulty range rather than to flatter the result:
**BigGAN** (native 128, so all four strategies are the identity on it - the study's control
row), **Midjourney** (the hardest cell, and the worst train/val gap anywhere), and **VQDM**
(middling). One arm, `center_crop`, because it is the non-resampling arm the headline rests on.

Regularization in both conditions: dropout 0.3, weight decay 1e-4 (AdamW), and early stopping
with patience 20. Patience is deliberately generous - the cosine schedule delivers most of its
gain as the learning rate decays to zero, so a small patience stops the run before the schedule
has paid out and looks like convergence. Watch `stopped_epoch` in the output: if it is far below
80, suspect the patience, not the model.

In [ ]:
SOURCES = ["BigGAN", "Midjourney", "VQDM"]
STRATEGY = "center_crop"
SEED = 0
EPOCHS = 80
PATIENCE = 20
DROPOUT = 0.3

base = (
    f'python -m src.train --cache-dir "{CACHE}" --strategy {STRATEGY} --seed {SEED}'
    f' --epochs {EPOCHS} --patience {PATIENCE} --dropout {DROPOUT}'
)
queue = []
for source in SOURCES:
    queue.append(("A ctrl ", f'{base} --source {source} --results-dir "{CTRL_DIR}"'))
    queue.append(("B d4+fc", f'{base} --source {source} --results-dir "{D4_DIR}" --d4 --fc-hidden 256'))

print(f"{len(queue)} runs queued\n")
started = time.perf_counter()
for label, cmd in queue:
    print(f"===== {label} {cmd.split('--source ')[1].split(' ')[0]}")
    !{cmd}
print(f"\nall done in {(time.perf_counter() - started) / 60:.1f} min")

## Read the result

The number that decides it is **off-domain mean**, not in-domain. Raising the diagonal while
cross-generator stands still is the outcome Gate 0 predicted, and it would not be worth the
full 28 runs.

Per-cell binomial standard error is +-1.6 pp, so a difference under ~2 pp on a single source is
not a difference. Look for a consistent sign across all three sources.

In [ ]:
def load(root):
    rows = {}
    for path in sorted(glob.glob(f"{root}/*/*/seed*/metrics.json")):
        m = json.loads(Path(path).read_text())
        rows[m["source"]] = m
    return rows

ctrl, exp = load(CTRL_DIR), load(D4_DIR)

print(f"{'source':<12}{'in-domain A':>13}{'B':>9}{'d':>8}   {'off-dom A':>10}{'B':>9}{'d':>8}   {'epochs A/B':>12}")
for source in SOURCES:
    a, b = ctrl.get(source), exp.get(source)
    if not (a and b):
        print(f"{source:<12}  missing - did that run fail?")
        continue
    print(
        f"{source:<12}{a['in_domain']:>13.4f}{b['in_domain']:>9.4f}{b['in_domain'] - a['in_domain']:>+8.4f}"
        f"   {a['off_domain_mean']:>10.4f}{b['off_domain_mean']:>9.4f}"
        f"{b['off_domain_mean'] - a['off_domain_mean']:>+8.4f}"
        f"   {a['config']['stopped_epoch']:>5}/{b['config']['stopped_epoch']:<6}"
    )

both = [(ctrl[s], exp[s]) for s in SOURCES if s in ctrl and s in exp]
if both:
    d_off = sum(b['off_domain_mean'] - a['off_domain_mean'] for a, b in both) / len(both)
    d_in = sum(b['in_domain'] - a['in_domain'] for a, b in both) / len(both)
    print(f"\nmean change: in-domain {d_in:+.4f}   off-domain {d_off:+.4f}")
    if d_off > 0.02:
        print("-> off-domain up by more than the per-cell SE. Worth the full 28 runs.")
    elif d_in > 0.02:
        print("-> diagonal only. This is what Gate 0 predicted; the full grid buys a nicer\n"
              "   in-domain number and no new claim.")
    else:
        print("-> no movement. Stop here and report it as a measured negative.")

## Hand off

Nothing above wrote into the repo. The metrics and checkpoints sit under
`.../deep_learning_results/<owner>/experiments/` in Drive; move whatever is worth keeping into
git yourself.

If you do promote these numbers, keep them in a **separate tree** from `results/runs` — the
four reported matrices are flip-only at 40 epochs, and mixing an 80-epoch D4 run into them
would silently corrupt every figure `analyze.py` produces.

In [ ]:
for name, root in (("A ctrl80", CTRL_DIR), ("B d4_fc256", D4_DIR)):
    metrics = glob.glob(f"{root}/*/*/seed*/metrics.json")
    ckpts = glob.glob(f"{root}/*/*/seed*/model.pt")
    size = sum(os.path.getsize(f) for f in metrics + ckpts) / 1e6
    print(f"{name:<12}{len(metrics)} metrics, {len(ckpts)} checkpoints, {size:.0f} MB")
print(f"\nin Drive at: {EXP}")